# 01 · Quality control — GSE216146

Builds the AnnData from the GEO counts files (sparse UMI matrix + gene list + cell metadata), keeps the six selected
samples (3 control `PN`, 3 chemotherapy `CN`), computes QC metrics from the raw counts, filters cells per sample and
genes after merging, then normalizes.

**Input** (in `$DATA`): `GSE216146_expression_matrix.mtx.gz` (cells x genes, integer counts),
`GSE216146_genes.tsv.gz`, `GSE216146_cell_metadata.tsv.gz` · **Output:** `results/01_qc.h5ad` (raw counts in
`layers["counts"]`, log-normalized `X`), QC plots in `results/plots/`, cell counts in `results/tables/01_qc_cell_counts.tsv`.

Notes on the input (checked on 2026-11-13):
- The mtx holds 19,834 cells x 32,285 genes; its rows follow the metadata order and its columns the gene list.
  It is identical to `raw.X` of the authors' h5ad (which is not used here).
- The cells were already filtered by the authors (≥ 500 genes, ribo < 25 %) and Scrublet doublets removed
  (`predicted_doublet` is all `False`). Only a few author columns are kept in `obs` (`qc.metadata_columns`);
  the author QC columns are dropped because QC is recomputed here.
- The authors' `pct_mt` (max 7 %) was computed on a matrix where `mt-` genes have zero counts, so their mitochondrial
  filter had no effect. Recomputed on the counts, the D20 batch has a much higher mt fraction (median ~11–14 % vs ~4–5 % in D21).
  A fixed 20 % cut would remove ~1/3 of the D20 cells, so the mt filter is **per sample: median + 3 MAD**.
- Mouse mitochondrial genes start with lower-case `mt-` (the original script used `Mt-`, which matched nothing).
- `pathology` has four codes (`PN`, `PS`, `CN`, `CS`); only `PN` and `CN` samples are analysed here.

In [ ]:
import gc
from importlib.metadata import version
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
qc = cfg["qc"]
samples = cfg["samples"]
scu.setup_plotting()
print("scanpy", version("scanpy"), "| anndata", version("anndata"))
print("samples:", samples)

## 1. Build the AnnData from the counts files

In [ ]:
# Streams the mtx and keeps only the selected samples (~10 s, ~1.3 GB RAM)
counts = scu.read_counts(cfg, samples)
assert counts.n_obs > 0
print(counts)
print("integer counts:", bool(np.all(counts.X.data == np.round(counts.X.data))))

In [ ]:
# Selected samples and their pathology code / batch (taken from the cell metadata)
design = (counts.obs[["biosample", "pathology", "batch"]]
          .value_counts().rename("n_cells").reset_index()
          .query("n_cells > 0").sort_values(["pathology", "biosample"]))
design

In [ ]:
# Check that each selected sample is present and has the expected pathology code
observed = design.set_index("biosample")["pathology"].astype(str)
mismatch = {s: (exp, observed.get(s)) for s, exp in samples.items() if observed.get(s) != exp}
assert not mismatch, f"sample/pathology mismatch: {mismatch}"
print("sample selection OK")

## 2. Per-sample QC
QC metrics are computed per sample from the raw counts, and cells are filtered with the thresholds in `config.yaml`.

In [ ]:
per_sample, removed = [], {}
for s in samples:
    a = counts[counts.obs["biosample"] == s].copy()
    scu.add_qc_metrics(a, qc)
    a.obs["qc_pass"] = False
    filtered, rm = scu.filter_cells(a, qc)
    a.obs.loc[filtered.obs_names, "qc_pass"] = True
    removed[s] = {"cells_before": a.n_obs, **rm, "cells_after": filtered.n_obs}
    per_sample.append(a)

pre = ad.concat(per_sample, join="inner", merge="same")
pre.obs["biosample"] = pre.obs["biosample"].astype(str).astype("category")
del per_sample, counts
gc.collect()

counts_table = pd.DataFrame(removed).T
counts_table.index.name = "biosample"
counts_table.insert(0, "pathology", [samples[s] for s in counts_table.index])
counts_table

In [ ]:
qc_cols = ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"]
pre.obs["qc"] = np.where(pre.obs["qc_pass"], "kept", "removed")

fig, axes = plt.subplots(1, len(qc_cols), figsize=(4 * len(qc_cols), 4))
for ax, col in zip(axes, qc_cols):
    sc.pl.violin(pre, col, groupby="biosample", jitter=0.3, size=0.5, rotation=90, ax=ax, show=False)
    ax.set_title(col)
fig.suptitle("Before filtering", y=1.02)
scu.save_fig(fig, cfg, "01_qc_violin_before")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sc.pl.scatter(pre, x="total_counts", y="n_genes_by_counts", color="qc", ax=ax, show=False, size=4)
scu.save_fig(fig, cfg, "01_qc_scatter_counts_genes")
plt.show()

## 3. Merge kept cells, filter genes, keep raw counts

In [ ]:
adata = pre[pre.obs["qc_pass"]].copy()
del pre
gc.collect()

n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=qc["min_cells"])
print(f"genes: {n_genes_before} -> {adata.n_vars} (min_cells={qc['min_cells']})")

# Condition labels applied once, here (PN -> Control, CN -> Chemotherapy)
adata.obs["condition"] = pd.Categorical(
    adata.obs["pathology"].astype(str).map(cfg["condition_labels"]),
    categories=[cfg["reference_condition"]] + [c for c in cfg["condition_labels"].values()
                                               if c != cfg["reference_condition"]])
adata.obs = adata.obs.drop(columns=["qc_pass", "qc"])
for col in adata.obs.select_dtypes("category"):
    adata.obs[col] = adata.obs[col].cat.remove_unused_categories()
adata

In [ ]:
fig, axes = plt.subplots(1, len(qc_cols), figsize=(4 * len(qc_cols), 4))
for ax, col in zip(axes, qc_cols):
    sc.pl.violin(adata, col, groupby="biosample", jitter=0.3, size=0.5, rotation=90, ax=ax, show=False)
    ax.set_title(col)
fig.suptitle("After filtering", y=1.02)
scu.save_fig(fig, cfg, "01_qc_violin_after")
plt.show()

In [ ]:
summary = (adata.obs.groupby(["condition", "biosample"], observed=True)
           .agg(cells=("n_genes_by_counts", "size"),
                median_genes=("n_genes_by_counts", "median"),
                median_counts=("total_counts", "median"),
                median_pct_mt=("pct_counts_mt", "median")))
summary

## 4. Normalize and save

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=qc["target_sum"])
sc.pp.log1p(adata)

out_h5ad = cfg["paths"]["results_dir"] / "01_qc.h5ad"
adata.write_h5ad(out_h5ad, compression="gzip")

out_table = cfg["paths"]["tables_dir"] / "01_qc_cell_counts.tsv"
counts_table.join(summary.reset_index("condition").drop(columns="cells")).to_csv(out_table, sep="\t")
print("wrote", out_h5ad, "and", out_table)
print(adata)

In [ ]:
del adata
gc.collect()